In [1]:
# Install dependencies
!pip install sentence-transformers nltk huggingface_hub -q

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

True

In [2]:
# Load HF Token dari Kaggle Secrets

from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")  # Pastikan nama secret-nya "HF_TOKEN"

print("✅ HF Token loaded")

✅ HF Token loaded


In [3]:
# Download artifacts dari HuggingFace Hub
from huggingface_hub import hf_hub_download

REPO_ID = "melisaolivia18/empathetic-retrieval-sbert"

path_embeddings  = hf_hub_download(repo_id=REPO_ID, filename="response_embeddings.npy", token=HF_TOKEN)
path_texts       = hf_hub_download(repo_id=REPO_ID, filename="response_texts.pkl",       token=HF_TOKEN)
path_emotions    = hf_hub_download(repo_id=REPO_ID, filename="emotion_labels.pkl",        token=HF_TOKEN)

print("✅ Artifacts downloaded")
print(f"   embeddings : {path_embeddings}")
print(f"   texts      : {path_texts}")
print(f"   emotions   : {path_emotions}")

response_embeddings.npy:   0%|          | 0.00/25.6M [00:00<?, ?B/s]

response_texts.pkl:   0%|          | 0.00/864k [00:00<?, ?B/s]

emotion_labels.pkl:   0%|          | 0.00/181k [00:00<?, ?B/s]

✅ Artifacts downloaded
   embeddings : /root/.cache/huggingface/hub/models--melisaolivia18--empathetic-retrieval-sbert/snapshots/2f307d214279bab300a171681e0a4175110d59f8/response_embeddings.npy
   texts      : /root/.cache/huggingface/hub/models--melisaolivia18--empathetic-retrieval-sbert/snapshots/2f307d214279bab300a171681e0a4175110d59f8/response_texts.pkl
   emotions   : /root/.cache/huggingface/hub/models--melisaolivia18--empathetic-retrieval-sbert/snapshots/2f307d214279bab300a171681e0a4175110d59f8/emotion_labels.pkl


In [4]:
# Load artifacts ke memori
import numpy as np
import pickle

response_embeddings = np.load(path_embeddings)          # shape: (N, 384)

with open(path_texts, "rb") as f:
    response_texts = pickle.load(f)                     # list[str], len N

with open(path_emotions, "rb") as f:
    emotion_labels = pickle.load(f)                     # list[str], len N

assert len(response_texts) == len(emotion_labels) == response_embeddings.shape[0], \
    "❌ Mismatch jumlah entri antar artifacts!"

print(f"✅ Artifacts loaded — total responses: {len(response_texts)}")
print(f"   Embedding shape : {response_embeddings.shape}")
print(f"   Emotion buckets : {set(emotion_labels)}")

✅ Artifacts loaded — total responses: 16661
   Embedding shape : (16661, 384)
   Emotion buckets : {'joyful', 'afraid', 'surprised', 't believe my daughter taught herself how to play the ukelele. i was amazed', 'sentimental', 'faithful', 'content', "m so mad with my brother. he stole from me and didn't think i would notice.", 'ashamed', 'terrified', 'prepared', '(', 'we were in a different country', 'apprehensive', 'trusting', 'lonely', 'nostalgic', 'devastated', 'proud', 'sad', 'guilty', 'excited', 'embarrassed', 'annoyed', 'disappointed', 'jealous', 'grateful', 'furious', 'angry', 'caring', 'anxious', 'confident', 'i really killed it!', 'hopeful', 't think i wold like super heroes', 't believe i like the show power so much. i was never really into shows like that', 'disgusted', 'anticipating', 'impressed'}


In [5]:
# Load Sentence-BERT model
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
print("✅ Sentence-BERT loaded: all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Sentence-BERT loaded: all-MiniLM-L6-v2


In [6]:
# 25 Test Cases (dari test-case.json)
test_cases = [
    # SADNESS
    {
        "emotion": "sadness",
        "input": "I feel so empty and lost today, like nothing I do matters anymore.",
        "reference": "It sounds like you're carrying a lot right now. It's okay to feel this way, and I'm here to listen."
    },
    {
        "emotion": "sadness",
        "input": "I've been crying all day and I don't even know why. I just feel so alone.",
        "reference": "Sometimes sadness doesn't need a reason. You're not alone in feeling this way, and it's okay to let it out."
    },
    {
        "emotion": "sadness",
        "input": "My best friend stopped talking to me and I miss them so much. It hurts every day.",
        "reference": "Losing someone close to you is really painful. It's natural to grieve that connection and miss them deeply."
    },
    {
        "emotion": "sadness",
        "input": "I feel like I'm invisible to everyone around me. No one notices when I'm struggling.",
        "reference": "Feeling unseen is one of the hardest things to go through. Your feelings are valid and you deserve to be heard."
    },
    {
        "emotion": "sadness",
        "input": "Everything feels grey and meaningless lately. I used to enjoy things but now nothing feels good.",
        "reference": "When joy feels out of reach, it can be really exhausting. I hear you, and I hope things start to feel lighter soon."
    },

    # ANGER
    {
        "emotion": "anger",
        "input": "I'm so furious right now. My coworker took credit for my work again and no one did anything.",
        "reference": "That sounds incredibly frustrating and unfair. It's completely understandable to feel angry when your efforts aren't recognized."
    },
    {
        "emotion": "anger",
        "input": "I can't believe people can be so selfish and inconsiderate. I'm done being patient.",
        "reference": "It makes sense that you're fed up. Constantly dealing with inconsiderate people is exhausting and it's okay to feel angry."
    },
    {
        "emotion": "anger",
        "input": "My parents keep making decisions for me without asking. I'm so tired of not being respected.",
        "reference": "Feeling like your autonomy isn't respected is really frustrating. Your need to be heard and treated as an adult is valid."
    },
    {
        "emotion": "anger",
        "input": "I worked so hard for this and someone else got it just because they know the right people. It's not fair.",
        "reference": "That kind of injustice is genuinely infuriating. Your hard work deserves recognition and it's okay to be upset about this."
    },
    {
        "emotion": "anger",
        "input": "I told them exactly what I needed and they ignored me completely. I feel disrespected.",
        "reference": "Being dismissed when you've clearly communicated your needs is deeply frustrating. Your feelings of anger are completely justified."
    },

    # JOY
    {
        "emotion": "joy",
        "input": "I just got accepted into my dream university! I can't stop smiling, I worked so hard for this.",
        "reference": "That is absolutely wonderful news! All your hard work paid off and you should be so proud of yourself."
    },
    {
        "emotion": "joy",
        "input": "I had the most amazing day with my family today. We laughed so much and it felt so warm.",
        "reference": "Those moments of genuine connection with family are so precious. It sounds like a truly beautiful day."
    },
    {
        "emotion": "joy",
        "input": "I finally finished the project I've been working on for months. I feel so accomplished and relieved.",
        "reference": "What an amazing feeling! Completing something you've dedicated so much time to is such a rewarding experience."
    },
    {
        "emotion": "joy",
        "input": "My partner surprised me with something really thoughtful today and I feel so loved and appreciated.",
        "reference": "That sounds so heartwarming. It's a beautiful feeling when someone who cares about you shows it in such a meaningful way."
    },
    {
        "emotion": "joy",
        "input": "I just found out I'm going to be an older sibling! Our family is growing and I'm beyond excited.",
        "reference": "What exciting news! A new addition to the family is such a joyful thing and your excitement is completely contagious."
    },

    # ANXIETY
    {
        "emotion": "anxiety",
        "input": "I have a big presentation tomorrow and I can't sleep. My mind keeps racing with everything that could go wrong.",
        "reference": "It's really common to feel this way before something important. Try to take slow breaths and remember that you've prepared for this."
    },
    {
        "emotion": "anxiety",
        "input": "I keep overthinking every conversation I have. I always feel like I said something wrong.",
        "reference": "Overthinking like that can be really exhausting. Most of the time, people are far less focused on our mistakes than we think."
    },
    {
        "emotion": "anxiety",
        "input": "I feel like something bad is about to happen even though nothing is wrong. I can't shake this feeling.",
        "reference": "That sense of dread without a clear cause can feel really unsettling. You're not alone in experiencing this kind of anxiety."
    },
    {
        "emotion": "anxiety",
        "input": "I have so many deadlines piling up and I don't know where to start. I feel paralyzed.",
        "reference": "Feeling overwhelmed by too many things at once is really difficult. Breaking things into smaller steps can sometimes help ease that paralyzed feeling."
    },
    {
        "emotion": "anxiety",
        "input": "I'm scared about the future and whether I'm making the right choices. What if I regret everything?",
        "reference": "Uncertainty about the future is something so many people struggle with. It's okay not to have everything figured out right now."
    },

    # CALM
    {
        "emotion": "calm",
        "input": "I just got back from a long walk in the park. I feel peaceful and clear-headed for the first time in weeks.",
        "reference": "That sounds really restorative. Time in nature has a way of quieting the mind and bringing things back into perspective."
    },
    {
        "emotion": "calm",
        "input": "I spent the morning journaling and meditating. I feel centered and ready to face the day.",
        "reference": "Starting the day with that kind of intention is really grounding. It sounds like you've found a routine that genuinely works for you."
    },
    {
        "emotion": "calm",
        "input": "Things have been stressful lately but today I just let myself rest and it feels good to do nothing.",
        "reference": "Rest is so important and it takes wisdom to allow yourself that space. It sounds like you really needed and deserved this today."
    },
    {
        "emotion": "calm",
        "input": "I had a quiet evening at home, made some tea, and read a book. I feel content and at ease.",
        "reference": "There's something really lovely about those simple, quiet evenings. It sounds like you gave yourself exactly what you needed."
    },
    {
        "emotion": "calm",
        "input": "I finally resolved a conflict with someone I care about and now I feel relieved and at peace.",
        "reference": "Resolving tension with someone you care about is such a relief. That sense of peace after reconciliation is really meaningful."
    },
]

print(f"✅ Test cases loaded: {len(test_cases)} cases")

✅ Test cases loaded: 25 cases


In [7]:
# Retrieval + BLEU Scoring per test case
from sklearn.metrics.pairwise import cosine_similarity
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.tokenize import word_tokenize

# Pre-build index per emotion bucket untuk efisiensi
emotion_labels_arr = np.array(emotion_labels)
unique_emotions    = list(set(emotion_labels))

bucket_index = {}
for emo in unique_emotions:
    mask = emotion_labels_arr == emo
    bucket_index[emo] = {
        "indices":    np.where(mask)[0],
        "embeddings": response_embeddings[mask],
        "texts":      [response_texts[i] for i in np.where(mask)[0]],
    }

smoother = SmoothingFunction().method1   # smoothing untuk kalimat pendek

results = []

for tc in test_cases:
    emotion   = tc["emotion"]
    user_input = tc["input"]
    reference  = tc["reference"]

    # — Encode input
    input_emb = model.encode([user_input])  # shape: (1, 384)

    # — Filter ke bucket emosi yang sesuai
    if emotion not in bucket_index:
        # fallback: cari di seluruh corpus jika bucket tidak ada
        bucket_embs  = response_embeddings
        bucket_texts = response_texts
        bucket_note  = "⚠️ fallback (no bucket)"
    else:
        bucket_embs  = bucket_index[emotion]["embeddings"]
        bucket_texts = bucket_index[emotion]["texts"]
        bucket_note  = f"bucket={emotion} ({len(bucket_texts)} responses)"

    # — Cosine similarity
    sims = cosine_similarity(input_emb, bucket_embs)[0]  # (N_bucket,)

    # — Ambil response dengan similarity tertinggi
    best_idx          = int(np.argmax(sims))
    best_sim          = float(sims[best_idx])
    retrieved_response = bucket_texts[best_idx]

    # — BLEU Score (unigram – 4-gram, smoothed)
    ref_tokens  = word_tokenize(reference.lower())
    hyp_tokens  = word_tokenize(retrieved_response.lower())
    bleu        = sentence_bleu([ref_tokens], hyp_tokens,
                                smoothing_function=smoother)

    results.append({
        "emotion":            emotion,
        "input":              user_input,
        "reference":          reference,
        "retrieved_response": retrieved_response,
        "cosine_similarity":  round(best_sim, 4),
        "bleu_score":         round(bleu, 4),
        "bucket_note":        bucket_note,
    })

print(f"✅ Retrieval + BLEU scoring selesai untuk {len(results)} test cases")

✅ Retrieval + BLEU scoring selesai untuk 25 test cases


In [8]:
# Agregasi & Tabel Hasil
import pandas as pd
from collections import defaultdict

# Tabel detail per test case
df = pd.DataFrame(results)[[
    "emotion", "input", "retrieved_response",
    "cosine_similarity", "bleu_score"
]]

df["input"]              = df["input"].str[:60] + "…"
df["retrieved_response"] = df["retrieved_response"].str[:70] + "…"

print("\n" + "="*80)
print("DETAIL RESULTS — 25 TEST CASES")
print("="*80)
print(df.to_string(index=True))

# Agregasi per emotion
agg = defaultdict(lambda: {"bleu_scores": [], "cosine_scores": []})
for r in results:
    agg[r["emotion"]]["bleu_scores"].append(r["bleu_score"])
    agg[r["emotion"]]["cosine_scores"].append(r["cosine_similarity"])

summary_rows = []
for emo, vals in sorted(agg.items()):
    summary_rows.append({
        "Emotion":          emo.capitalize(),
        "N":                len(vals["bleu_scores"]),
        "Avg BLEU":         round(sum(vals["bleu_scores"])   / len(vals["bleu_scores"]),   4),
        "Avg Cosine Sim":   round(sum(vals["cosine_scores"]) / len(vals["cosine_scores"]), 4),
    })

# Baris overall
all_bleu   = [r["bleu_score"]        for r in results]
all_cosine = [r["cosine_similarity"] for r in results]
summary_rows.append({
    "Emotion":        "**OVERALL**",
    "N":              len(results),
    "Avg BLEU":       round(sum(all_bleu)   / len(all_bleu),   4),
    "Avg Cosine Sim": round(sum(all_cosine) / len(all_cosine), 4),
})

df_summary = pd.DataFrame(summary_rows)

print("\n" + "="*60)
print("SUMMARY — PER EMOTION + OVERALL")
print("="*60)
print(df_summary.to_string(index=False))


DETAIL RESULTS — 25 TEST CASES
    emotion                                                          input                                                       retrieved_response  cosine_similarity  bleu_score
0   sadness  I feel so empty and lost today, like nothing I do matters an…  Whats wrong? Did anything happen? Have you reached out to any friends …             0.6254      0.0078
1   sadness  I've been crying all day and I don't even know why. I just f…                                 That's too bad. Why were you all alone?…             0.6850      0.0068
2   sadness  My best friend stopped talking to me and I miss them so much…                            I miss mine too. Where did you guys grow up?…             0.6588      0.0106
3   sadness  I feel like I'm invisible to everyone around me. No one noti…                                    I understand that feeling sometimes.…             0.5970      0.0029
4   sadness  Everything feels grey and meaningless lately. I used to enjo

In [9]:
# Export ke CSV
df_full = pd.DataFrame(results)
df_full.to_csv("retrieval_bleu_results.csv", index=False)
df_summary.to_csv("retrieval_bleu_summary.csv", index=False)

print("\n✅ Exported: retrieval_bleu_results.csv & retrieval_bleu_summary.csv")


✅ Exported: retrieval_bleu_results.csv & retrieval_bleu_summary.csv
